# 04: WSI Preprocessing (ABMIL Embeddings)

Loads pre-computed ABMIL patient embeddings and prepares them as a feature matrix for LUAD vs LUSC classification.

ABMIL (Attention-Based Multiple Instance Learning) embeddings are produced by a gated attention network trained on the TCGA cohort. The attention mechanism learns to weight morphologically informative tiles more heavily than uninformative background, producing a 512-dimensional patient-level representation. All 512 dimensions are retained as the embedding is already compact and task-specific.

**Pipeline**
| Step | Action |
|------|--------|
| 1 | Load ABMIL embeddings from `.pt` file |
| 2 | Load labels and train/test split |
| 3 | Align patients across modalities |
| 4 | PCA visualisation: LUAD vs LUSC separation |
| 5 | Save final feature matrix |

> **Input:** `Processed/wsi_embeddings_abmil.pt` — dict of {patient_id: tensor(512)}  
> **Output:** `wsi_features.csv`, (N_patients × 512)

## Imports and configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
from pathlib import Path
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Paths
import os

BASE_DIR = Path(
    os.environ.get("FUSION_REPO_DIR", str(Path.cwd()))
).expanduser().resolve()

if BASE_DIR.name in {"Preprocessing", "Training"}:
    BASE_DIR = BASE_DIR.parent

if not (BASE_DIR / "Training" / "07_controlled_early_late_fusion.ipynb").is_file():
    raise RuntimeError(
        "Start Jupyter from the repository directory or set FUSION_REPO_DIR."
    )
DATA_DIR = Path(
    os.environ.get("FUSION_DATA_DIR", str(BASE_DIR / "Data"))
).expanduser().resolve()
PROCESSED = Path(
    os.environ.get("FUSION_PROCESSED_DIR", str(DATA_DIR.parent / "Processed"))
).expanduser().resolve()

OUT_DIR = BASE_DIR / "Preprocessing"
FIG_DIR    = OUT_DIR / 'Figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

ABMIL_PATH  = PROCESSED / 'wsi_embeddings_abmil.pt'
LABELS_PATH = DATA_DIR / "multiomics_labels.tsv"
SPLIT_PATH  = DATA_DIR  / 'train_test_split.csv'
OUT_PATH    = OUT_DIR   / 'wsi_features.csv'

In [ ]:
# Config
RANDOM_STATE = 42

print('Imports OK')
print(f'ABMIL embeddings : {ABMIL_PATH}')
print(f'Output           : {OUT_PATH}')

## Load ABMIL embeddings

In [ ]:
emb_dict = torch.load(ABMIL_PATH, weights_only=False)

# Convert to DataFrame — patients × 512
emb_data = {
    pid: tensor.numpy() if isinstance(tensor, torch.Tensor) else np.array(tensor)
    for pid, tensor in emb_dict.items()
}
df_wsi = pd.DataFrame.from_dict(emb_data, orient='index')
df_wsi.index = df_wsi.index.str.strip().str.upper().str[:12]
df_wsi.index.name = 'patient_id'
df_wsi.columns = [f'abmil_{i}' for i in range(df_wsi.shape[1])]

print(f'ABMIL embeddings loaded : {df_wsi.shape}  (patients × features)')
print(f'Embedding dimension     : {df_wsi.shape[1]}')
print(f'Value range             : [{df_wsi.values.min():.4f}, {df_wsi.values.max():.4f}]')
print(f'NaNs                    : {df_wsi.isnull().sum().sum()}')

## 2. Load labels and train/test split

In [ ]:
# Labels
df_labels = pd.read_csv(LABELS_PATH, sep='\t', index_col='patient_id')
df_labels.index = df_labels.index.str.strip().str.upper().str[:12]
labels = df_labels['subtype_simple'].map({'LUAD': 0, 'LUSC': 1})

assert SPLIT_PATH.exists(), f'Split file not found: {SPLIT_PATH}. Run notebook 01 first.'
split_df  = pd.read_csv(SPLIT_PATH, index_col='patient_id')
split_df.index = split_df.index.str.strip().str.upper().str[:12]

print(f'Labels loaded : {len(labels)} patients')
print(f'Split loaded  : train={(split_df["split"]=="train").sum()}, test={(split_df["split"]=="test").sum()}')

## 3. Align patients across modalities

In [ ]:
common_ids = df_wsi.index.intersection(labels.index).intersection(split_df.index)
df_wsi     = df_wsi.loc[common_ids]
y          = labels.loc[common_ids]
split_df   = split_df.loc[common_ids]

train_ids  = split_df[split_df['split'] == 'train'].index
test_ids   = split_df[split_df['split'] == 'test'].index

print(f'Common patients : {len(common_ids)}')
print(f'Train           : {len(train_ids)}')
print(f'Test            : {len(test_ids)}')
print(f'LUAD: {(y==0).sum()}, LUSC: {(y==1).sum()}')

## 4. PCA visualization: LUAD vs LUSC separation

PCA fitted on training patients only, applied to all patients.
Strong separation in 2D confirms the ABMIL embeddings carry
discriminative morphological signal between LUAD and LUSC.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca.fit(df_wsi.loc[train_ids])
coords = pca.transform(df_wsi)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(coords[y==0, 0], coords[y==0, 1],
           c='steelblue', alpha=0.5, s=20, label='LUAD')
ax.scatter(coords[y==1, 0], coords[y==1, 1],
           c='tomato', alpha=0.5, s=20, label='LUSC')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title('PCA of ABMIL WSI embeddings (512-dim → 2D)', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'wsi_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'PC1 + PC2 variance explained : {sum(pca.explained_variance_ratio_):.1%}')
print(f'Figure saved to {FIG_DIR}/wsi_pca.png')

### 4a. Explained variance per component

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(df_wsi.loc[train_ids])

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95   = int(np.searchsorted(cumvar, 0.95)) + 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, color='steelblue', linewidth=2)
ax.axhline(0.95, color='orange', linestyle=':', linewidth=1.5, label='95% variance')
ax.axvline(n_95, color='red',    linestyle='--', linewidth=1.5,
           label=f'{n_95} components')
ax.set_xlabel('Number of PCA components', fontsize=12)
ax.set_ylabel('Cumulative explained variance', fontsize=12)
ax.set_title('Cumulative explained variance: ABMIL WSI embeddings', fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(FIG_DIR / 'wsi_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'{n_95} components explain 95% of variance in ABMIL embeddings')
print(f'Figure saved to {FIG_DIR}/wsi_pca_variance.png')

In [ ]:
from sklearn.decomposition import PCA

# Reduce WSI to 50 PCA components (train-fitted only)
pca_wsi = PCA(n_components=50, random_state=RANDOM_STATE)
pca_wsi.fit(df_wsi.loc[train_ids])
wsi_reduced = pca_wsi.transform(df_wsi)

df_final = pd.DataFrame(
    wsi_reduced,
    index=df_wsi.index,
    columns=[f'wsi_pc_{i}' for i in range(wsi_reduced.shape[1])]
)

print(f'Variance explained by 50 components: {pca_wsi.explained_variance_ratio_.sum():.2%}')

In [ ]:
from sklearn.decomposition import PCA

pca_wsi = PCA(n_components=100, random_state=RANDOM_STATE)
pca_wsi.fit(df_wsi.loc[train_ids])
wsi_reduced = pca_wsi.transform(df_wsi)

df_final = pd.DataFrame(
    wsi_reduced,
    index=df_wsi.index,
    columns=[f'wsi_pc_{i}' for i in range(100)]
)
print(f'Variance explained by 100 components: {pca_wsi.explained_variance_ratio_.sum():.2%}')

In [ ]:
df_final.to_csv(OUT_PATH)
print(f'Saved feature matrix to {OUT_PATH}')
print(f'Shape                : {df_final.shape}  (patients × features)')
print(f'NaNs                 : {df_final.isnull().sum().sum()}')

## Summary

| Step | Result |
|------|--------|
| Source | `wsi_embeddings_abmil.pt` |
| Raw patients | 829 |
| Patients after alignment | 829 |
| Train / Test | 581 / 250 |
| Embedding dimension | 512 |
| Components for 95% variance | 3 |
| **Final features** | **512 (all ABMIL dimensions retained)** |
| Output file | `wsi_features.csv` |
| Feature selection | None since ABMIL embeddings are compact and task-specific |
| Leakage prevention | PCA fitted on train patients only |